# PolyNet: a hands-on tour — homopolymers

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Biomaterials-for-Medical-Devices-AI/PolyNet/blob/main/notebooks/PolyNet_homopolymers.ipynb)

[**PolyNet**](https://github.com/Biomaterials-for-Medical-Devices-AI/PolyNet) is a Python package for polymer informatics: it
predicts polymer properties with graph neural networks (GNNs) and traditional machine
learning (TML), and it explains those predictions in chemical terms.

> **If you use PolyNet, please cite the paper:** Aguilar-Bejarano, E.; Alexander, M. R.;
> Winkler, D. A.; Figueredo, G. *PolyNet: A Python Package for Polymer Informatics.*
> ChemRxiv **2026**. doi:[10.26434/chemrxiv.15010453/v1](https://doi.org/10.26434/chemrxiv.15010453/v1)

> **This is the homopolymers notebook.** Its sibling,
> [`PolyNet_workshop_copolymers.ipynb`](https://github.com/Biomaterials-for-Medical-Devices-AI/PolyNet/blob/main/notebooks/PolyNet_workshop_copolymers.ipynb),
> walks through the same workflow for **copolymers**. Everything below works the same way in
> both; only the dataset and a couple of settings differ.

This notebook walks through the whole workflow on a real dataset, one stage at a time:

| Section | What it covers |
|---|---|
| 1 | **Your data** — a built-in benchmark, or your own CSV |
| 2 | **Configuration** — the settings objects every stage takes |
| 3 | **Representation** — polymer graphs and descriptor vectors |
| 4 | **Splitting** — repeated train/validation/test splits |
| 5 | **GNN training** — training an ensemble, predicting, scoring |
| 6 | **TML training** — random forest and XGBoost on the same splits |
| 7 | **Comparing models** — metrics across splits |
| 8 | **The experiment folder** — what PolyNet saves |
| 9 | **Deployment** — predicting properties of new, unseen polymers |
| 10 | **Applicability domain** — when *not* to trust a prediction |
| 11 | **Explainability** — which fragments and which descriptors drive a prediction |
| 12 | **Other ways to run PolyNet** — the GUI and the YAML/CLI pipeline |

**How to use it:** run the cells in order, top to bottom. Every cell has an explanation above
it saying what the step does and why it matters. The whole notebook takes roughly 10–15
minutes on a free Colab CPU runtime — no GPU needed.

> No machine learning experience is assumed. The dataset size, the number of splits and the
> number of training epochs are deliberately small so everything finishes while you watch.
> Section 0.4 has a single cell where you can scale them up afterwards.

---

## 0. Setup

### 0.1 Install PolyNet from PyPI

PolyNet is published on PyPI, so one `pip install` brings in everything: RDKit, PyTorch,
PyTorch Geometric, scikit-learn, XGBoost, SHAP and the rest. It takes **2–4 minutes** and
pulls in a few hundred MB.

`--ignore-requires-python` is there because PolyNet declares `python >=3.11,<3.12` while
Colab currently runs Python 3.12. Every one of PolyNet's dependencies supports 3.12 and
PolyNet's own self-check passes there, so the flag is safe — and harmless on Python 3.11.

If Colab tells you to restart the session afterwards, click **Restart session**, then carry
on from section 0.3 — you do not need to re-run the install.

In [ ]:
%pip install -q --ignore-requires-python polynet

### 0.2 Optional: the PSMILES canonicaliser

The built-in datasets are written in **PSMILES** — SMILES with two `[*]` attachment points
marking where the repeat unit connects, e.g. `*CC(*)c1ccccc1` for polystyrene.

Canonicalising PSMILES (so the same repeat unit is always written the same way) needs
`canonicalize-psmiles`, a package from the Kuenneth group that is not on PyPI. PolyNet
installs it for you with the command below.

> **Licence:** `canonicalize-psmiles` is distributed by Georgia Tech Research Corporation
> under a licence that permits **non-commercial use only**. `--yes` accepts it. If you would
> rather not install it, skip this cell and set `CANONICALISE = False` in section 1.1 — the
> notebook will run either way, it just will not canonicalise the structures.

In [ ]:
!polynet install-psmiles --yes

### 0.3 Check the installation

`polynet check` trains tiny GNN and TML models on synthetic data, explains them and predicts
on new data — an end-to-end test that the installation really works. It takes about 30
seconds and is the fastest way to tell a broken environment from a broken config later on.

In [ ]:
!polynet check

### 0.4 Imports, logging and workshop scale

Three things in one cell.

**Logging.** PolyNet's core is deliberately free of any UI: every stage reports progress
through the standard `logging` module and installs **no handler of its own**, so everything
informative — split sizes, model names, files written, stages that were skipped and why — is
invisible by default. Several stages degrade gracefully (they skip a step and log why) rather
than raising, so without a handler you can silently miss the explanation. Adding one is worth
doing before anything else.

**Imports.** Everything PolyNet exposes publicly comes from a handful of modules:
`polynet.data` (loading and preparing datasets), `polynet.config` (typed settings and paths),
`polynet.pipeline` (the pipeline stages), and `polynet.plotting` / `polynet.visualization`
(plots).

**Scale.** The constants at the end are the **only knobs you need to turn** to make this
notebook bigger. The defaults are chosen so that a full run fits in a coffee break; the
comments say what to use for real work.

In [ ]:
import logging
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import polynet
from polynet.config.enums import (
    AtomFeature,
    BondFeature,
    DatasetName,
    DescriptorMergingMethod,
    FeatureSelection,
    MolecularDescriptor,
    Network,
    ProblemType,
    SplitMethod,
    SplitSampler,
    SplitType,
    StringRepresentation,
    TargetTransformDescriptor,
    TraditionalMLModel,
    TrainingParam,
    TransformDescriptor,
)

# --- logging: send PolyNet's INFO messages to the notebook ------------------
handler = logging.StreamHandler(sys.stdout)
handler.setFormatter(logging.Formatter("%(levelname)-8s %(message)s"))

polynet_logger = logging.getLogger("polynet")
polynet_logger.handlers.clear()
polynet_logger.addHandler(handler)
polynet_logger.setLevel(logging.INFO)
polynet_logger.propagate = False   # do not also print through the root logger

print("PolyNet version:", polynet.__version__)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
warnings.filterwarnings("ignore", category=UserWarning)

# ---------------------------------------------------------------------------
# Workshop scale - the only numbers you need to change to scale this up
# ---------------------------------------------------------------------------
N_SAMPLES = 500      # polymers used for modelling (the full dataset has 7,367)
N_HELD_OUT = 50      # polymers kept completely aside, used as "new" data in section 9
N_BOOTSTRAPS = 3     # repeated train/val/test splits (use 5-10 for real work)
EPOCHS = 80          # GNN training epochs per split (use 250+ for real work)
RANDOM_SEED = 42     # one seed controls splits, model init and HPO

### 0.5 Where results are saved, and how to keep them

PolyNet writes each experiment into one folder. Locally that is
`~/PolyNetExperiments/<experiment name>`, the same place the GUI looks — which is why an
experiment built in this notebook can be reopened in the app.

On Colab the saving code works exactly the same, but **nothing in a Colab runtime survives
it**: when the runtime disconnects or is recycled the whole virtual machine is deleted. So
the question is not whether saving works, it is where you want the files to end up. Set
`MOUNT_DRIVE = True` below and your Google Drive is mounted and experiments are written
straight into it, so results outlive the session (you will be asked to authorise access).
Leave it `False` and they go to `/content/PolyNetExperiments`, which appears in the **Files**
pane on the left — in that case remember section 8.2 before you close the tab.

In [ ]:
MOUNT_DRIVE = False   # set to True to keep every result in your Google Drive

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and MOUNT_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    EXPERIMENTS_DIR = Path("/content/drive/MyDrive/PolyNetExperiments")
elif IN_COLAB:
    # Under /content so the folder shows up in the Files pane on the left.
    EXPERIMENTS_DIR = Path("/content/PolyNetExperiments")
else:
    from polynet.config.paths import polynet_experiments_base_dir

    # ~/PolyNetExperiments - the folder the GUI reads.
    EXPERIMENTS_DIR = polynet_experiments_base_dir()

EXPERIMENTS_DIR.mkdir(parents=True, exist_ok=True)
print("Running in Colab:  ", IN_COLAB)
print("Experiments go to: ", EXPERIMENTS_DIR)

### 0.6 The options you can change

PolyNet has far more choices than one notebook can use, and the cell below prints **all of
them** — every architecture, descriptor, atom feature, sampler, scaler, fragmentation and plot
type — written exactly as you would type it, grouped by the section that uses it.

Nothing in this cell affects the run. Think of it as the menu: find an option you like, go to the
section named in its heading, and swap it in. You will meet the same options again beside the
settings themselves — every configuration cell lists its alternatives in comments, and the blocks
that are commented out are there to be uncommented. Each of those sections also ends with a short
**Try it** suggestion, so if you only have time for one experiment, start there.

The list is generated from PolyNet itself rather than typed out, so it cannot go out of date —
and it can tell you things a written list could not, such as which samplers are deterministic and
which one is disabled.

In [ ]:
import textwrap

from polynet.config.enums import (
    ApplicabilityDomainMetric,
    AttributionPlotType,
    FragmentationMethod,
    ImportanceNormalisationMethod,
    Pooling,
    ShapGlobalPlotType,
)
from polynet.config.schemas.split_data import (
    DETERMINISTIC_SAMPLERS,
    DISABLED_SAMPLERS,
    SAMPLERS_USING_FINGERPRINTS,
    SAMPLERS_USING_TARGET,
)


def typed(enum):
    """Every member of an enum, written the way you would type it."""
    return [f"{enum.__name__}.{member.name}" for member in enum]


def show(setting, items, note=None):
    print(f"\n  {setting}")
    print(textwrap.fill("  ".join(items), width=94, initial_indent=" " * 6,
                        subsequent_indent=" " * 6))
    if note:
        print(textwrap.fill(note, width=94, initial_indent=" " * 6, subsequent_indent=" " * 6))


def heading(text):
    print(f"\n{'=' * 94}\n{text}\n{'=' * 94}")


heading("SECTION 1 - the task itself (DataConfig)")
show("problem_type", typed(ProblemType),
     note="note: this notebook's modelling cells are written for regression.")

heading("SECTION 3 - turning each polymer into numbers (RepresentationConfig)")
show("node_features - atom properties that become graph features", typed(AtomFeature))
show("edge_features - bond properties", typed(BondFeature))
show("molecular_descriptors - the vectors the traditional models learn from",
     typed(MolecularDescriptor),
     note="rdkit and morgan are used below. polybert and polymetrix are polymer-specific and "
          "download a model the first time; dataframe and rdkit_dataframe use columns you "
          "already have in your table.")
show("smiles_merge_approach - how several monomers become one representation",
     typed(DescriptorMergingMethod))

heading("SECTION 4 - how the data is split (SplitConfig)")
print("\n  sampler - which polymers end up in which set")
for sampler in SplitSampler:
    tags = []
    if sampler in DISABLED_SAMPLERS:
        tags.append("DISABLED")
    if sampler in SAMPLERS_USING_FINGERPRINTS:
        tags.append("fingerprint")
    if sampler in SAMPLERS_USING_TARGET:
        tags.append("uses the target")
    if sampler in DETERMINISTIC_SAMPLERS:
        tags.append("deterministic")
    suffix = f"   [{', '.join(tags)}]" if tags else ""
    print(f"      SplitSampler.{sampler.name}{suffix}")
print(textwrap.fill(
    "fingerprint = places polymers in chemical space for a deliberately hard test set. "
    "deterministic = every repeat gives the same split, so the spread across splits collapses. "
    "DISABLED = cannot be used for new splits.",
    width=94, initial_indent=" " * 6, subsequent_indent=" " * 6))
show("split_type", typed(SplitType),
     note="only train_val_test is implemented: repeated random train/validation/test "
          "splits, set by test_ratio, val_ratio and n_bootstrap_iterations.")
show("split_method", typed(SplitMethod),
     note="stratified keeps each class's proportions, so it is for classification only.")

heading("SECTION 5 - graph neural networks (TrainGNNConfig)")
show("architectures - one model is trained per architecture per split", typed(Network))
show("pooling - how node embeddings become one vector per polymer", typed(Pooling))
show("hyperparameters shared by every architecture", typed(TrainingParam),
     note="plus per-architecture keys: embedding_dim, n_convolutions, readout_layers, dropout, "
          "and extras such as improved (GCN). An unknown key raises an error listing what that "
          "architecture does accept.")
show("target transform (TargetTransformConfig, section 5.1)",
     typed(TargetTransformDescriptor))

heading("SECTION 6 - traditional models (TrainTMLConfig, FeatureTransformConfig)")
show("models - each is trained on every descriptor set", typed(TraditionalMLModel),
     note="linear_regression is regression-only and logistic_regression classification-only; "
          "the rest handle both.")
show("scaler - applied to the descriptor matrix", typed(TransformDescriptor))
show("selectors - which descriptor columns to drop", typed(FeatureSelection),
     note="variance and correlation are the two that are implemented; pass an empty "
          "dict to select nothing. no_selection and lasso are not accepted yet.")

heading("SECTION 10 - applicability domain (ApplicabilityDomainConfig)")
show("metric - how distance to the training set is measured",
     typed(ApplicabilityDomainMetric))

heading("SECTION 11 - explainability (ExplainabilityConfig, TMLExplainabilityConfig)")
show("fragmentation - how a polymer is cut into pieces", typed(FragmentationMethod))
show("plot_type - the GNN attribution figure", typed(AttributionPlotType))
show("plot_type - the SHAP figure", typed(ShapGlobalPlotType))
show("normalisation - how attributions are scaled before they are compared",
     typed(ImportanceNormalisationMethod))

print(f"\n{'=' * 94}")
print("Every one of these is an ordinary Python value: change it in the cell whose section is")
print("named above, then re-run from that cell onwards.")
print("=" * 94)

---

## 1. Your data

### 1.1 Choose a dataset

Two options, and the cell below is the only place you choose between them.

**Option A — a built-in benchmark (the default).** PolyNet ships two datasets so you can start
without hunting for data:

| Name | What it is | Task |
|---|---|---|
| `curated_tg` | 7,367 homopolymers (PSMILES) with experimental glass transition temperatures, from the curated [PolyMetriX](https://github.com/lamalab-org/polymetrix) dataset | regression on `Tg(K)` |
| `fluorine_mri_snr` | 418 fluorinated copolymers of six acrylate monomers with measured ¹⁹F MRI signal-to-noise ratios | regression on `SNR` |

This notebook uses `curated_tg`; the copolymer sibling notebook uses the other one.

**Option B — your own CSV.** Set `USE_MY_OWN_DATA = True` and name your columns. In Colab,
leave `MY_CSV_PATH` empty and you will be asked to upload the file when you run the next
cell; running locally, put the path there instead. Your CSV needs one column with the polymer
structure (SMILES or PSMILES), one column with the property to predict, and ideally a column
of unique IDs.

**Everything after this cell reads your choices from these variables.** With your columns
named correctly, the rest of the notebook runs on your data with no further edits.

In [ ]:
# ===========================================================================
#  CHOOSE YOUR DATASET
#  Leave USE_MY_OWN_DATA = False to run the built-in Tg benchmark.
# ===========================================================================
USE_MY_OWN_DATA = False

CANONICALISE = True   # set to False if you skipped `polynet install-psmiles` in section 0.2

# --- only read when USE_MY_OWN_DATA = True ---------------------------------
MY_CSV_PATH = ""                # leave empty in Colab to upload the file instead
MY_SMILES_COLS = ["SMILES"]     # the column holding the polymer structure
MY_TARGET_COL = "Tg"            # the property to predict
MY_ID_COL = "polymer_id"        # unique identifier, or None to number the rows
MY_PROBLEM_TYPE = "regression"  # see the note below about classification
MY_TARGET_NAME = "Tg"           # short name used in plot labels and result columns
MY_TARGET_UNITS = "K"           # may be an empty string

### 1.2 Load it, and check it

`load_benchmark_dataset` downloads (first time only), caches and **validates** the built-in
data; `load_dataset` does the same for a CSV of your own. Either way you get a table indexed by
the ID column, and a `columns` object that records which column is which — every later section
reads that object rather than hardcoding a name.

For your own data the cell also runs a few checks before PolyNet sees it, so a mistake comes
back as a sentence rather than a traceback from deep inside the package: column names that are
not in the CSV, a target that is not numeric, missing targets or structures, structures RDKit
cannot read, and repeated IDs.

> **A note on classification.** The modelling cells below are wired for **regression**. If your
> target is a label rather than a number, PolyNet handles that too, but four things need
> changing: `num_classes` and `class_names` in section 2, the target transform in section 5.1
> (drop it), `show_continuous_distribution` in section 1.5 (use `show_label_distribution`
> instead), and the parity plot in section 5.3 (`plot_results_stage` writes ROC curves and
> confusion matrices for you instead). Setting `MY_PROBLEM_TYPE = "classification"` stops with
> that same reminder rather than failing somewhere deeper.

In [ ]:
from polynet.data.creator import (
    BENCHMARK_COLUMNS,
    BenchmarkColumns,
    held_out_dataset,
    subsample_dataset,
)
from polynet.data.loader import load_benchmark_dataset, load_dataset


def stop(problem, fix):
    """Fail with an explanation instead of a traceback from deep inside PolyNet."""
    raise ValueError(f"{problem}\n  -> {fix}")


def check_columns(frame, wanted):
    missing = [c for c in wanted if c and c not in frame.columns]
    if missing:
        stop(
            f"These columns are not in the CSV: {missing}",
            f"Fix the names in the cell above. The CSV has: {list(frame.columns)}",
        )


def check_target(frame, column):
    if frame[column].isna().any():
        stop(
            f"{int(frame[column].isna().sum())} row(s) have no value in '{column}'",
            "Remove those rows - a model cannot learn from a missing target.",
        )
    numbers = pd.to_numeric(frame[column], errors="coerce")
    unreadable = frame.loc[numbers.isna(), column]
    if len(unreadable):
        stop(
            f"{len(unreadable)} value(s) in '{column}' are not numbers, for example "
            f"{list(unreadable.unique()[:3])}",
            "Clean those rows, or use a classification task if the target is a label.",
        )


def check_structures(frame, smiles_cols):
    from rdkit import Chem, RDLogger

    RDLogger.DisableLog("rdApp.*")   # we report the failures ourselves, in plain words
    try:
        for column in smiles_cols:
            blank = frame[column].isna() | (frame[column].astype(str).str.strip() == "")
            if blank.any():
                stop(
                    f"{int(blank.sum())} row(s) have no structure in '{column}'",
                    "Remove those rows, or fill in the missing structures.",
                )
            bad = [
                str(index)
                for index, value in frame[column].items()
                if Chem.MolFromSmiles(str(value)) is None
            ]
            if bad:
                stop(
                    f"RDKit cannot read {len(bad)} structure(s) in '{column}' "
                    f"(rows {bad[:5]}{' ...' if len(bad) > 5 else ''})",
                    "Check those SMILES - an unbalanced ring or bracket is the usual cause.",
                )
    finally:
        RDLogger.EnableLog("rdApp.*")


def check_ids(frame, column):
    if column is None:
        return
    if frame[column].isna().any():
        stop(
            f"Some rows have no value in the ID column '{column}'",
            "Fill them in, or set MY_ID_COL = None to number the rows instead.",
        )
    repeated = frame[column][frame[column].duplicated()].unique()
    if len(repeated):
        stop(
            f"The ID column '{column}' repeats {len(repeated)} value(s), for example "
            f"{list(repeated[:3])}",
            "IDs must be unique, or set MY_ID_COL = None to number the rows instead.",
        )


if not USE_MY_OWN_DATA:
    BENCHMARK = DatasetName.CuratedTg
    columns = BENCHMARK_COLUMNS[BENCHMARK]
    DATA_FILE_NAME = "curated_tg_workshop.csv"
    full_dataset = load_benchmark_dataset(
        BENCHMARK,
        smiles_cols=columns.smiles_cols,
        target_col=columns.target_col,
        id_col=columns.id_col,
        problem_type=columns.problem_type,
    )
else:
    BENCHMARK = None   # DataConfig records the CSV path instead
    if MY_PROBLEM_TYPE != "regression":
        stop(
            "This notebook's modelling cells are wired for regression.",
            "See the note above for the four things to change for a classification target.",
        )

    csv_path = Path(MY_CSV_PATH) if MY_CSV_PATH else None
    if csv_path is None and IN_COLAB:
        from google.colab import files

        print("Choose your CSV:")
        uploaded = files.upload()          # Colab writes the file into the working directory
        csv_path = Path(next(iter(uploaded)))
    if csv_path is None or not csv_path.exists():
        stop(
            f"There is no CSV to read (MY_CSV_PATH = {MY_CSV_PATH!r})",
            "Give the path to a CSV, or run this in Colab and upload one.",
        )

    raw = pd.read_csv(csv_path)
    check_columns(raw, [*MY_SMILES_COLS, MY_TARGET_COL, MY_ID_COL])
    check_target(raw, MY_TARGET_COL)
    check_structures(raw, MY_SMILES_COLS)
    check_ids(raw, MY_ID_COL)

    columns = BenchmarkColumns(
        smiles_cols=MY_SMILES_COLS,
        id_col=MY_ID_COL,
        target_col=MY_TARGET_COL,
        problem_type=ProblemType(MY_PROBLEM_TYPE),
        target_name=MY_TARGET_NAME,
        target_units=MY_TARGET_UNITS,
    )
    DATA_FILE_NAME = csv_path.name
    full_dataset = load_dataset(
        path=csv_path,
        smiles_cols=columns.smiles_cols,
        target_col=columns.target_col,
        id_col=columns.id_col,
        problem_type=columns.problem_type,
    )

# Axis labels and titles throughout: "Tg (K)", or just "SNR" when there are no units.
units_suffix = f" ({columns.target_units})" if columns.target_units else ""


def with_units(name):
    return f"{name}{units_suffix}"


print("Dataset:      ", "your CSV" if USE_MY_OWN_DATA else BENCHMARK.value)
print("Structures in:", columns.smiles_cols)
print("Predicting:   ", f"{columns.target_col} as {with_units(columns.target_name)}")
print("Polymers:     ", len(full_dataset))
full_dataset.head()

### 1.3 Take a workshop-sized subset

`subsample_dataset` draws a reproducible random subset, and `held_out_dataset` takes rows
that are guaranteed **not** to be in that subset. We use the subset for everything up to
section 8, and keep the held-out polymers as genuinely unseen data for the deployment and
applicability-domain sections — exactly how you would treat a batch of newly designed
polymers.

In [ ]:
data = subsample_dataset(full_dataset, n_samples=N_SAMPLES, seed=RANDOM_SEED)
new_polymers = held_out_dataset(
    full_dataset, n_samples=N_SAMPLES, n_held_out=N_HELD_OUT, seed=RANDOM_SEED
)

print(f"Modelling set: {len(data)} polymers")
print(f"Held out as 'new' polymers: {len(new_polymers)} polymers")
print(f"Overlap between them: {len(set(data.index) & set(new_polymers.index))}")

data[[columns.target_col]].describe().T

### 1.4 Validate and canonicalise the structures

`prepare_structures` is the single door every entry point goes through — the GUI, the CLI and
this notebook. It **detects** whether the structure columns are SMILES or PSMILES (PSMILES if
every entry carries at least two `*` attachment points), **validates** every structure with
RDKit, and **canonicalises** them so the same repeat unit always gets the same string.

Doing this before featurising means a typo in your CSV fails loudly here, rather than
silently becoming a weird graph later.

In [ ]:
from polynet.data.structures import prepare_structures

data, representation = prepare_structures(
    data,
    smiles_cols=columns.smiles_cols,
    representation=None,          # None = detect it from the structures
    canonicalise=CANONICALISE,
)

print("Detected / used representation:", representation.value)
data.head(3)

### 1.5 Look at the target before you model it

Always. A long tail, a hard floor, a suspicious spike at zero or two separated populations
all change which model and which target transform make sense.

`show_continuous_distribution` returns a Matplotlib figure, so you can style or save it. For
a classification target, use `show_label_distribution` instead — it annotates the count of
each class, which is how you spot imbalance.

In [ ]:
from polynet.plotting.data_analysis import show_continuous_distribution

fig = show_continuous_distribution(
    data,
    target_variable=columns.target_col,
    bins=30,
    title=f"{with_units(columns.target_name)} of the modelled polymers",
)
plt.show()

### 1.6 Draw a few polymers

A quick visual check that the structures are what you think they are. The `*` atoms are the
attachment points of the repeat unit — RDKit draws them as dummy atoms.

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw

sample = data.head(6)
mols = [Chem.MolFromSmiles(s) for s in sample[columns.smiles_cols[0]]]
Draw.MolsToGridImage(
    mols,
    molsPerRow=3,
    subImgSize=(260, 200),
    legends=[
        f"ID {i} - {columns.target_name} {t:.3g}{units_suffix}"
        for i, t in zip(sample.index, sample[columns.target_col])
    ],
)

---

## 2. Configuration: typed objects, not loose dictionaries

Every PolyNet stage takes a **settings object**, never a raw dictionary. That is a deliberate
design choice: a typo, a missing field or an impossible combination (a regression task with
three classes) raises a clear error *before* any training starts, and the same objects
serialise to JSON so an experiment records exactly how it was run.

The ones you will meet here: `DataConfig` (dataset, structure columns, target, task type),
`RepresentationConfig` (graph features, descriptors, how monomers are merged), `SplitConfig`
(split strategy, ratios, repeats), `TrainGNNConfig` and `TrainTMLConfig` (which models to
train), `FeatureTransformConfig` and `TargetTransformConfig` (scaling), `ExplainabilityConfig`
(attribution settings) and `ApplicabilityDomainConfig` (section 10).

### 2.1 The data config and the experiment folder

An **experiment** is a folder. Everything PolyNet computes — representations, split indices,
trained models, predictions, metrics, plots, explanations — is written there, and
`polynet.config.paths` has a helper for every single path, so nothing is hardcoded.

Notice that every field below comes from the choices you made in section 1.1, so this cell
needs no editing whether you are running the benchmark or your own CSV.

In [ ]:
from polynet.config.io import load_options, save_options
from polynet.config.paths import (
    data_file_path,
    data_options_path,
    data_spliting_options_path,
    general_options_path,
    ml_results_file_path,
    model_metrics_file_path,
    plots_directory,
    polynet_experiment_path,
    preprocessing_tml_model_options_path,
    representation_options_path,
    split_indices_path,
    target_transform_options_path,
    train_gnn_model_options_path,
    train_tml_model_options_path,
)
from polynet.config.schemas import DataConfig, GeneralConfig
from polynet.experiment.manager import create_experiment

EXPERIMENT_NAME = "workshop_homopolymers"
# EXPERIMENTS_DIR was chosen in section 0.5. Locally it is what
# polynet_experiment_path(EXPERIMENT_NAME) returns: ~/PolyNetExperiments/workshop_homopolymers.
experiment_path = EXPERIMENTS_DIR / EXPERIMENT_NAME

data_cfg = DataConfig(
    data_name=DATA_FILE_NAME,
    benchmark_dataset=BENCHMARK,             # None when you brought your own CSV
    data_path=str(data_file_path(DATA_FILE_NAME, experiment_path)),
    smiles_cols=columns.smiles_cols,
    canonicalise_smiles=CANONICALISE,
    target_variable_col=columns.target_col,
    problem_type=columns.problem_type,
    string_representation=representation,    # detected in section 1.4
    id_col=columns.id_col,
    num_classes=1,                           # 1 for regression, N for N-class classification
    target_variable_name=columns.target_name,
    target_variable_units=columns.target_units,
)

# Creates the folder and saves data_options.json inside it.
create_experiment(experiment_path, data_cfg)

# The experiment-wide settings. Every GUI page after Create Experiment reads this
# file, so an experiment built from the API has to write it too.
general_cfg = GeneralConfig(
    name=EXPERIMENT_NAME, output_dir=str(experiment_path), random_seed=RANDOM_SEED
)
save_options(general_options_path(experiment_path), general_cfg)

# The dataset itself is saved too: later stages (and the applicability domain)
# read it back from here.
data.to_csv(data_file_path(data_cfg.data_name, experiment_path))

print("Experiment folder:", experiment_path)
print("Data options saved to:", data_options_path(experiment_path))

---

## 3. Representation: from a string to numbers

A model cannot read a SMILES string, so PolyNet turns each polymer into numbers in two
complementary ways, and you can use either or both.

**Graphs** (for GNNs) — atoms are nodes, bonds are edges, and you choose which RDKit properties
become features. Each feature is one-hot encoded over the `allowable_vals` you list;
`wildcard: True` adds one extra "anything else" bucket so an unexpected element does not break
the encoding.

**Descriptor vectors** (for traditional models) — RDKit descriptors, count fingerprints
(Morgan, RDKit), polyBERT embeddings, PolyMetriX polymer descriptors, or columns you already
have in your table.

One `RepresentationConfig` holds both. `IsAttachmentPoint` deserves a note: for PSMILES it
drops the `[*]` dummy atoms from the graph and instead flags the real atoms they were bonded
to — so the model learns *where* the chain continues without carrying meaningless dummy nodes.

**Try it:** add `MolecularDescriptor.PolyBERT` for a learned embedding of the whole repeat unit,
or `MolecularDescriptor.PolyMetriX` for polymer-specific descriptors — both download a model the
first time you use them. Or add an atom feature from the catalogue in section 0.6
(`GetHybridization` and `IsInRing` are good ones) and watch the node-feature count change when the
graphs are built in 3.1.

In [ ]:
from polynet.config.schemas import RepresentationConfig

repr_cfg = RepresentationConfig(
    # Only one monomer column here, so there is nothing to merge.
    smiles_merge_approach=DescriptorMergingMethod.NoMerging,
    # --- graph representation, used by the GNNs ---
    # A one-hot feature takes "allowable_vals" (the categories) and "wildcard" (True adds an
    # "other" bucket for everything not listed). A yes/no or single-number feature takes {}.
    node_features={
        AtomFeature.GetAtomicNum: {
            "allowable_vals": [6, 7, 8, 9, 14, 15, 16, 17, 35],  # C N O F Si P S Cl Br
            "wildcard": True,
        },
        AtomFeature.GetTotalNumHs: {"allowable_vals": [0, 1, 2, 3], "wildcard": True},
        AtomFeature.GetIsAromatic: {"allowable_vals": [True, False], "wildcard": False},
        AtomFeature.IsAttachmentPoint: {},  # PSMILES: flags the atoms the [*] were bonded to
        # Every other atom property PolyNet can encode. Uncomment to add it to every node:
        # AtomFeature.GetTotalDegree: {"allowable_vals": [1, 2, 3, 4], "wildcard": True},
        # AtomFeature.GetImplicitValence: {"allowable_vals": [0, 1, 2, 3], "wildcard": True},
        # AtomFeature.GetFormalCharge: {"allowable_vals": [-1, 0, 1], "wildcard": True},
        # AtomFeature.IsInRing: {},
        # AtomFeature.GetMass: {},  # one number per atom (mass / 100), not a one-hot block
        # AtomFeature.GetHybridization: {   # the categories are RDKit members, not strings
        #     "allowable_vals": [
        #         Chem.rdchem.HybridizationType.SP,
        #         Chem.rdchem.HybridizationType.SP2,
        #         Chem.rdchem.HybridizationType.SP3,
        #     ],
        #     "wildcard": True,
        # },
        # AtomFeature.GetChiralTag: {
        #     "allowable_vals": [
        #         Chem.rdchem.ChiralType.CHI_UNSPECIFIED,
        #         Chem.rdchem.ChiralType.CHI_TETRAHEDRAL_CW,
        #         Chem.rdchem.ChiralType.CHI_TETRAHEDRAL_CCW,
        #     ],
        #     "wildcard": True,
        # },
    },
    edge_features={
        BondFeature.GetBondTypeAsDouble: {
            "allowable_vals": [1.0, 1.5, 2.0, 3.0],  # single, aromatic, double, triple
            "wildcard": True,
        },
        # Every other bond property, same rule - {} for the yes/no ones:
        # BondFeature.GetIsAromatic: {},
        # BondFeature.GetIsConjugated: {},
        # BondFeature.IsInRing: {},
        # BondFeature.GetStereo: {
        #     "allowable_vals": [
        #         Chem.rdchem.BondStereo.STEREONONE,
        #         Chem.rdchem.BondStereo.STEREOE,
        #         Chem.rdchem.BondStereo.STEREOZ,
        #     ],
        #     "wildcard": True,
        # },
    },
    # --- descriptor representation, used by the traditional models ---
    # Each entry is its own feature matrix, so every traditional model in section 6 is
    # trained once per entry here, and section 11.2 gets one SHAP figure per entry.
    molecular_descriptors={
        MolecularDescriptor.RDKit: "all",                            # every RDKit descriptor
        MolecularDescriptor.Morgan: {"fp_size": 1024, "radius": 2},  # count fingerprint
        # "all" can be a list of names instead: ["MolWt", "TPSA", "MolLogP"]
        # A fingerprint takes True for its defaults (2048 bins, Morgan radius 3), or a dict.
        # The rest of the menu - uncomment one to train on it as well:
        # MolecularDescriptor.RDKitFP: {"fp_size": 2048},  # RDKit path count fingerprint
        # MolecularDescriptor.PolyBERT: True,              # polymer language model (one download)
        # MolecularDescriptor.PolyMetriX: {                # polymer-aware descriptors
        #     "backbone": ["num_rotatable_bonds", "num_rings", "molecular_weight"],
        #     "side_chain": ["num_rotatable_bonds", "num_rings", "molecular_weight"],
        #     "agg": ["sum", "mean"],
        # },
        # MolecularDescriptor.DataFrame: ["Mw", "PDI"],    # numeric columns of your own CSV
    },
    # Columns of your own CSV to append to every descriptor set above *and* to the GNN's
    # graph embedding - a measured property the model may see. Never the target column.
    # polymer_descriptors=["Mw"],
    weights_col=None,  # homopolymers: no monomer ratios
)

save_options(representation_options_path(experiment_path), repr_cfg)
print("Node features:", [f.value for f in repr_cfg.node_features])
print("Edge features:", [f.value for f in repr_cfg.edge_features])
print("Descriptor sets:", [d.value for d in repr_cfg.molecular_descriptors])

### 3.1 Build both representations

`build_graph_dataset` returns a PyTorch Geometric dataset: one graph per polymer, each with a
node feature matrix `x`, the connectivity `edge_index`, the edge features `edge_attr` and the
target `y`. `compute_descriptors` builds one table per descriptor set, saves each as a CSV
under `representation/Descriptors/` and returns them sanitised and ready for training.

One PSMILES detail matters here: before computing descriptors, PolyNet **caps each `[*]`
attachment point with a hydrogen**. A dummy atom has no mass, valence or electronegativity, so
leaving it in place corrupts many descriptors — the Gasteiger-charge ones return `NaN`
outright. Capping turns `[*]CC[*]` into ethane and gives physically meaningful values.

In [ ]:
from polynet.pipeline import build_graph_dataset, compute_descriptors

graph_dataset = build_graph_dataset(data, data_cfg, repr_cfg, experiment_path)

graph = graph_dataset[0]
print(f"Graphs in dataset:   {len(graph_dataset)}")
print(f"Node features:       {graph.num_node_features}")
print(f"Edge features:       {graph.num_edge_features}")
print(f"First graph:         {graph}")
print(f"  atoms (nodes):     {graph.num_nodes}")
print(f"  bonds (edges / 2): {graph.num_edges // 2}")
print(f"  target y:          {float(graph.y):.3g}\n")

descriptor_dfs = compute_descriptors(data, data_cfg, repr_cfg, experiment_path)
for name, frame in descriptor_dfs.items():
    print(f"{name:10s} {frame.shape[0]} polymers x {frame.shape[1] - 1} features")

descriptor_dfs[MolecularDescriptor.RDKit].iloc[:5, :6]

---

## 4. Splitting: one split is never enough

A single train/test split gives you one number and no idea how much it would move with a
different split. PolyNet therefore repeats the split `n_bootstrap_iterations` times, trains one
model per repeat, and reports the **distribution** of each metric. Every section after this one
works with that ensemble of models rather than a single model.

`SplitConfig` also chooses *how* samples are assigned, and `sampler` is the interesting knob:
`random` is an honest baseline, while `kennard_stone`, `spxy` and `kmeans` place each polymer by
its fingerprint and build a deliberately **hard**, chemically distinct test set — a far better
estimate of how a model will do on genuinely new chemistry.

The split indices are saved to `split_indices.json`, so the exact splits can be audited or
reused — including by the applicability domain in section 10.

**Try it:** swap `SplitSampler.Random` for `SplitSampler.KennardStone` and watch the test metrics
fall. That drop is the honest measure of generalisation to new chemistry, and it is the single
most informative one-word change in this notebook. `SplitSampler.TargetProperty` is harder still:
it sorts by the property itself, so the model has to extrapolate beyond the range it was trained
on. Both are deterministic — repeating them gives the same split every time, so the spread across
splits collapses, which section 0.6 flags.

In [ ]:
from polynet.config.schemas import SplitConfig
from polynet.pipeline import compute_data_splits

split_cfg = SplitConfig(
    split_type=SplitType.TrainValTest,  # the only one implemented so far
    split_method=SplitMethod.Random,    # or SplitMethod.Stratified, for classification
    # Random | KennardStone | SPXY | KMeans | TargetProperty (OptiSim is disabled).
    # Section 0.6 tags which ones place polymers by fingerprint and which are
    # deterministic; the fingerprint they use is set by sampling_fingerprint.
    sampler=SplitSampler.Random,   # try SplitSampler.KennardStone for a harder test set
    # sampling_fingerprint={"fingerprint": "morgan", "fp_size": 2048, "radius": 3},
    test_ratio=0.15,
    val_ratio=0.15,                # -> 70 / 15 / 15
    n_bootstrap_iterations=N_BOOTSTRAPS,
    train_set_balance=None,        # binary classification only: undersample the majority
)

# The GUI reads the split settings back on the training, explain and results pages.
save_options(data_spliting_options_path(experiment_path), split_cfg)

train_idxs, val_idxs, test_idxs = compute_data_splits(
    data=data,
    data_cfg=data_cfg,
    split_cfg=split_cfg,
    random_seed=RANDOM_SEED,
    out_dir=experiment_path,
    weights_col=repr_cfg.weights_col,
)
split_indexes = (train_idxs, val_idxs, test_idxs)

for i, (tr, va, te) in enumerate(zip(train_idxs, val_idxs, test_idxs), start=1):
    print(f"Split {i}: train {len(tr)}, validation {len(va)}, test {len(te)}")
print("\nSaved to:", split_indices_path(experiment_path))

---

## 5. Training graph neural networks

PolyNet implements six architectures: `GCN`, `GAT`, `GraphSAGE`, `MPNN`, `CGCNN` and
`TransformerConvGNN`. You list the ones you want in `gnn_convolutional_layers` together with
their hyperparameters, and PolyNet trains one model per architecture per split. Hyperparameter
keys come from the `TrainingParam` and architecture enums, so they are checked: an unknown key
raises an error listing what *is* allowed, instead of being silently ignored.

We train a single GCN here to keep the run short. For real work you rarely know the right
hyperparameters up front — leave an architecture's block empty (`{}`) and PolyNet searches them
for you with Ray Tune + ASHA, which is powerful but far slower than a workshop allows.

### 5.1 Target scaling

`TargetTransformConfig` scales the target for training and inverse-transforms the predictions
afterwards, so every metric and plot you see is back in the original units. The scaler is
fitted on the training samples of each split only, never the test ones. This matters whenever
the target sits far from zero or spans orders of magnitude — a target in the hundreds makes
gradient descent needlessly hard, and `Log10` or `Log1p` are the better choice for a target
spanning decades.

**Try it:** add a second architecture and PolyNet trains it on the same splits, with section 7
comparing them for you. Copy the `Network.GCN` block, change the key to `Network.GAT` or
`Network.GraphSAGE`, and drop the `improved` line (that one is GCN-specific) — the other seven
keys work for every architecture. Keep the hyperparameters explicit: an **empty** block means
"search them with Ray Tune", which is far slower than a workshop allows. Changing `pooling` to
`Pooling.GlobalMeanMaxPool` is a one-word experiment too.

In [ ]:
from polynet.config.schemas import TargetTransformConfig, TrainGNNConfig

# StandardScaler | MinMaxScaler | RobustScaler | Log10 | Log1p | NoTransformation.
# Fitted on each split's training rows and undone before anything is reported, so the
# metrics and plots stay in the target's own units.
target_cfg = TargetTransformConfig(strategy=TargetTransformDescriptor.StandardScaler)
save_options(target_transform_options_path(experiment_path), target_cfg)

gnn_cfg = TrainGNNConfig(
    train_gnn=True,
    epochs=EPOCHS,
    gnn_convolutional_layers={
        Network.GCN: {
            TrainingParam.LearningRate: 0.005,  # how big a step each batch takes
            TrainingParam.BatchSize: 32,        # polymers per step
            "embedding_dim": 64,                # width of every hidden layer
            "n_convolutions": 3,                # message-passing layers = how far a node sees
            "readout_layers": 2,                # dense layers after pooling
            "dropout": 0.1,                     # 0 for no dropout
            # how the node embeddings become one vector per polymer:
            # global_mean_pool | global_max_pool | global_add_pool | global_mean_max_pool
            "pooling": "global_mean_pool",
            "improved": False,                  # GCN only: self-loops weighted 2 instead of 1
        },
        # The other five architectures PolyNet ships. They take the same keys as above;
        # only the last line differs. Uncomment one and it is trained too - one model per
        # architecture per split - then compared in section 7 and explained in section 11.
        #   Network.GAT             "num_heads": 4   graph attention
        #   Network.TransformerGNN  "num_heads": 4   transformer-style (dot-product) attention
        #   Network.GraphSAGE       "bias": True     sample and aggregate
        #   Network.MPNN            -                edge-conditioned message passing (NNConv)
        #   Network.CGCNN           -                crystal-graph convolutions (CGConv)
        # GCN and GraphSAGE ignore the edge features from section 3; the other four use them,
        # so if you added bond features there, one of those four is the fairer test.
        # Network.GAT: {
        #     TrainingParam.LearningRate: 0.005,
        #     TrainingParam.BatchSize: 32,
        #     "embedding_dim": 64,
        #     "n_convolutions": 3,
        #     "readout_layers": 2,
        #     "dropout": 0.1,
        #     "pooling": "global_mean_pool",
        #     "num_heads": 4,
        # },
        # An empty block hands that architecture to Ray Tune, which searches its
        # hyperparameters for you instead (much slower, and it writes gnn_hyp_opt/):
        # Network.MPNN: {},
    },
    # Optional: optimiser, learning-rate schedule and loss, shared by every architecture.
    # The defaults are shown. Uncommenting needs two more imports:
    #   from polynet.config.enums import Optimizer, RegressionLoss, Scheduler
    #   from polynet.config.schemas.training import GNNOptimisationConfig
    # optimisation=GNNOptimisationConfig(
    #     optimizer=Optimizer.Adam,                   # Adam | SGD | RMSprop | Adadelta | Adagrad
    #     scheduler=Scheduler.ReduceLROnPlateau,      # or StepLR | MultiStepLR | ExponentialLR
    #     scheduler_factor=0.9,
    #     scheduler_patience=15,
    #     regression_loss=RegressionLoss.RMSE,        # RMSE | MSE | MAE
    # ),
)

save_options(train_gnn_model_options_path(experiment_path), gnn_cfg)
print("Architectures to train:", [n.value for n in gnn_cfg.gnn_convolutional_layers])
print("Epochs per model:", gnn_cfg.epochs)
print("Models that will be trained:", len(gnn_cfg.gnn_convolutional_layers) * N_BOOTSTRAPS)

### 5.2 Train, predict, score

Three calls, and they are the same three for every model type in PolyNet:

1. `train_gnn` trains one model per architecture per split and saves each as a `.pt` file in
   `ml_results/models/`. It returns the models, their data loaders and the fitted target
   scalers.
2. `run_gnn_inference` runs every model over its own train/validation/test sets and returns
   one tidy DataFrame of predictions, inverse-transformed back to the original units.
3. `compute_metrics` turns that DataFrame into `{split: {model: {set: {metric: value}}}}`.

That predictions DataFrame, shown at the end of the cell, is the backbone of everything
downstream — plots, metrics, the applicability domain. Its column names are predictable:
`True Tg`, `GCN Predicted Tg`, plus `Index`, `Set` and `bootstrap_iteration`.

This is the slowest cell in the notebook — roughly 1–3 minutes at the default scale.

In [ ]:
from polynet.pipeline import compute_metrics, run_gnn_inference, train_gnn

gnn_models, gnn_loaders, gnn_target_scalers = train_gnn(
    graph_dataset,
    split_indexes,
    data_cfg,
    gnn_cfg,
    random_seed=RANDOM_SEED,
    out_dir=experiment_path,
    target_cfg=target_cfg,
)

gnn_predictions = run_gnn_inference(
    trained_models=gnn_models,
    loaders=gnn_loaders,
    data_cfg=data_cfg,
    split_cfg=split_cfg,
    target_scalers=gnn_target_scalers,
)

gnn_metrics = compute_metrics(gnn_predictions, gnn_models, data_cfg, split_cfg)

print("Trained models:", list(gnn_models))
print()
for split, models in gnn_metrics.items():
    for model, sets in models.items():
        test = {str(k): round(v, 3) for k, v in sets["Test"].items() if v is not None}
        print(f"split {split} | {model} | test: {test}")

gnn_predictions.head()

### 5.3 A parity plot

`plot_parity` is the standard regression check: predicted against true, with the ideal
`y = x` line. Colouring by `Set` shows at a glance whether the model generalises or has simply
memorised the training polymers.

In [ ]:
from polynet.config.column_names import (
    get_iterator_name,
    get_predicted_label_column_name,
    get_true_label_column_name,
)
from polynet.config.constants import ResultColumn
from polynet.visualization.plots import plot_parity

iterator = get_iterator_name(split_cfg.split_type)
true_col = get_true_label_column_name(data_cfg.target_variable_name)
pred_col = get_predicted_label_column_name(data_cfg.target_variable_name, "GCN")

first_split = gnn_predictions[gnn_predictions[iterator] == "1"]

fig = plot_parity(
    y_true=first_split[true_col],
    y_pred=first_split[pred_col],
    hue=first_split[ResultColumn.SET],
    title="GCN, split 1",
    x_label=f"True {with_units(data_cfg.target_variable_name)}",
    y_label=f"Predicted {with_units(data_cfg.target_variable_name)}",
)
plt.show()

---

## 6. Training traditional models on the same splits

GNNs are not automatically the right answer. On a few hundred polymers, a random forest on
RDKit descriptors is often competitive and trains in seconds — and because PolyNet runs both on
*identical* splits, the comparison is fair. Each model is trained on every descriptor set, so
two models × two descriptor sets × three splits = twelve models here, named
`{model}-{descriptor}_{split}`.

Descriptor matrices need care first: RDKit gives you ~200 columns on wildly different scales,
and a 1024-bit fingerprint is mostly zeros. `FeatureTransformConfig` sets the scaler and the
feature selectors, fitted on the training set of each split only — `variance` drops
near-constant columns, `correlation` drops one of each pair of near-duplicate descriptors.
Expect a warning about a dozen dropped columns: RDKit cannot compute the Gasteiger-charge and
`BCUT2D_*` descriptors for some polymers and returns `NaN`, and any column with `NaN` or
infinity is dropped before fitting, so a few unusable descriptors cannot poison a whole model.

One flag is worth understanding. `include_validation_in_training=True` (the default) means the
traditional models train on train + validation, because they do not need a validation set for
early stopping the way GNNs do. That is more data and a stronger model, but it means the
validation rows are *training data* for them — so keep the **test** set as the only fair
comparison between the two families, and set the flag to `False` if you want them trained on
exactly the same rows.

**Try it:** add `TraditionalMLModel.LinearRegression` as a baseline — the fastest way to find out
whether your problem needs anything clever at all — or `TraditionalMLModel.SupportVectorMachine`.
`FeatureSelection.Lasso` is worth a look on descriptor sets this wide.

In [ ]:
from polynet.config.schemas import FeatureTransformConfig, TrainTMLConfig

preprocessing_cfg = FeatureTransformConfig(
    # StandardScaler | MinMaxScaler | RobustScaler | PowerTransformer | QuantileTransformer
    # | Normalizer | NoTransformation. Fitted on each split's training rows.
    scaler=TransformDescriptor.StandardScaler,
    selectors={
        FeatureSelection.Variance: {"threshold": 0.05},   # drop near-constant columns
        FeatureSelection.Correlation: {"threshold": 0.95},  # drop near-duplicate columns
        # Those two are the implemented selectors; pass {} to select nothing.
    },
    random_state=RANDOM_SEED,
)

tml_cfg = TrainTMLConfig(
    train_tml=True,
    include_validation_in_training=True,  # TML models train on train + validation
    # Each model here is trained on every descriptor set from section 3, once per split.
    # The keys inside a block are that estimator's own scikit-learn / XGBoost arguments.
    selected_models={
        TraditionalMLModel.RandomForest: {"n_estimators": 200, "max_depth": 10},
        TraditionalMLModel.XGBoost: {"n_estimators": 200, "max_depth": 6, "learning_rate": 0.05},
        # The rest of the menu:
        # TraditionalMLModel.SupportVectorMachine: {"C": 1.0, "kernel": "rbf"},
        # TraditionalMLModel.LinearRegression: {},      # regression only
        # TraditionalMLModel.LogisticRegression: {},    # classification only
        # An empty block means "search the hyperparameters for me" (RandomizedSearchCV over
        # the grid in polynet/config/search_grid.py), so {} above is HPO, not defaults.
    },
)

save_options(preprocessing_tml_model_options_path(experiment_path), preprocessing_cfg)
save_options(train_tml_model_options_path(experiment_path), tml_cfg)
print("TML models:", [m.value for m in tml_cfg.selected_models])

In [ ]:
from polynet.pipeline import run_tml_inference, train_tml

tml_models, tml_training_data, tml_scalers, tml_target_scalers = train_tml(
    desc_dfs=descriptor_dfs,
    split_indexes=split_indexes,
    data_cfg=data_cfg,
    tml_cfg=tml_cfg,
    preprocessing_cfg=preprocessing_cfg,
    random_seed=RANDOM_SEED,
    out_dir=experiment_path,
    target_cfg=target_cfg,
)

tml_predictions = run_tml_inference(
    trained=tml_models,
    training_data=tml_training_data,
    data_cfg=data_cfg,
    split_cfg=split_cfg,
    target_scalers=tml_target_scalers,
)

tml_metrics = compute_metrics(tml_predictions, tml_models, data_cfg, split_cfg)

print("Trained models:", list(tml_models))
print()
for split, models in tml_metrics.items():
    for model, sets in models.items():
        test = {str(k): round(v, 3) for k, v in sets["Test"].items() if v is not None}
        print(f"split {split} | {model:28s} | test: {test}")

---

## 7. Comparing models honestly

Because every model was trained on the same repeated splits, you can compare the
*distribution* of test metrics rather than single numbers. The table below collects them; the
boxplot shows the spread across splits with the mean and its 95 % confidence interval
overlaid.

With only three splits the intervals are wide — that is the point. If two models' intervals
overlap heavily, you do not have evidence that one is better. For publication-grade
comparisons, raise `N_BOOTSTRAPS` and use the paired statistics in
`polynet.utils.statistical_analysis`.

In [ ]:
rows = []
for source in (gnn_metrics, tml_metrics):
    for split, models in source.items():
        for model, sets in models.items():
            for set_name, metrics in sets.items():
                for metric, value in metrics.items():
                    if value is None:
                        continue
                    rows.append(
                        {
                            "model": model,
                            "split": split,
                            "set": set_name,
                            "metric": str(metric),
                            "value": value,
                        }
                    )

all_metrics = pd.DataFrame(rows)

test_summary = (
    all_metrics[all_metrics["set"] == "Test"]
    .pivot_table(index="model", columns="metric", values="value", aggfunc=["mean", "std"])
    .round(3)
)
test_summary

In [ ]:
from polynet.visualization.plots import plot_bootstrap_boxplots

rmse_by_model = (
    all_metrics[(all_metrics["set"] == "Test") & (all_metrics["metric"] == "rmse")]
    .groupby("model")["value"]
    .apply(list)
    .to_dict()
)

fig = plot_bootstrap_boxplots(
    rmse_by_model,
    metric_name=f"Test RMSE{units_suffix}",
    title="Test RMSE across splits",
)
plt.show()

---

## 8. Saving the experiment

So far the predictions live in memory. Writing them into the experiment folder is not
housekeeping — later stages *read them from there*. In particular, the applicability domain in
section 10 calibrates expected error against the held-out test errors it finds in
`ml_results/predictions.csv`. `plot_results_stage` also generates the standard result plots
(parity plots, learning curves, and for classification ROC curves and confusion matrices) and
saves them as PNGs.

In [ ]:
import json

from polynet.pipeline import plot_results_stage

# GNN and TML predictions share the Index / iterator columns, so they merge into one table.
merged_predictions = pd.merge(
    gnn_predictions.drop(columns=[true_col]),
    tml_predictions.drop(columns=[ResultColumn.SET]),
    on=[ResultColumn.INDEX, iterator],
)
merged_predictions.to_csv(ml_results_file_path(experiment_path), index=False)

combined_metrics = {}
for source in (gnn_metrics, tml_metrics):
    for split, models in source.items():
        combined_metrics.setdefault(split, {}).update(models)

with open(model_metrics_file_path(experiment_path), "w") as handle:
    json.dump(combined_metrics, handle, indent=2, default=str)

plot_results_stage(
    gnn_predictions, gnn_models, data_cfg, split_cfg, plots_directory(experiment_path)
)
plot_results_stage(
    tml_predictions, tml_models, data_cfg, split_cfg, plots_directory(experiment_path)
)

print("Predictions:", ml_results_file_path(experiment_path))
print("Metrics:    ", model_metrics_file_path(experiment_path))
print("Plots:      ", plots_directory(experiment_path))

### 8.1 What an experiment folder looks like

Everything needed to reproduce, audit, deploy or reopen the experiment in the GUI is in one
directory, and models are plain `.pt` (PyTorch) and `.joblib` (scikit-learn/XGBoost) files, so
nothing is locked inside PolyNet. The `*_options.json` files are the settings from section 2 —
which is how the deployment and explainability stages know how the models were built without
you telling them again.

The summary below groups files by folder, because PyTorch Geometric caches every individual
graph as its own `.pt` file under `representation/GNN/processed/` — useful for fast reloading,
far too noisy to list one by one.

In [ ]:
from collections import defaultdict

by_folder = defaultdict(list)
for path in sorted(experiment_path.rglob("*")):
    if path.is_file():
        by_folder[path.parent.relative_to(experiment_path)].append(path)

for folder, files in sorted(by_folder.items(), key=lambda item: str(item[0])):
    total_kb = sum(f.stat().st_size for f in files) / 1024
    print(f"{str(folder) + '/':42s} {len(files):5d} files {total_kb:10.1f} KB")
    shown = files if len(files) <= 8 else files[:6]
    for f in shown:
        print(f"    {f.name}")
    if len(files) > 8:
        print(f"    ... and {len(files) - 6} more")

### 8.2 Taking the experiment with you (Colab only)

**Running locally, there is nothing to do here** — the folder printed above is permanent, and
`polynet` opens it in the GUI directly. The cell below just tells you where it is.

**On Colab** the runtime is deleted when you disconnect, so if you did not mount Drive in
section 0.5 this is how you leave with your results: the cell zips the experiment folder and
starts a browser download. Unzip it into `~/PolyNetExperiments/` on your own machine and the GUI
will list it like any other experiment. Run this cell again at the end of the notebook to pick
up the explanations and predictions produced in sections 9 to 11.

The graph cache under `representation/GNN/processed/` is one `.pt` file per polymer. It is a
*cache*, rebuilt by PyTorch Geometric the next time you featurise the same data, so it is left
out of the archive by default to keep it readable — set `INCLUDE_GRAPH_CACHE = True` for a
byte-for-byte copy.

In [ ]:
INCLUDE_GRAPH_CACHE = False   # True for a byte-for-byte copy, graph cache included

if not IN_COLAB:
    print("Nothing to download - your experiment is already a permanent folder:")
    print("   ", experiment_path)
    print("\nOpen it with `polynet` (the GUI), or copy the folder wherever you want it.")
else:
    import zipfile

    from google.colab import files

    # /content is where the Files pane on the left looks.
    archive = Path("/content") / f"{EXPERIMENT_NAME}.zip"

    skipped = 0
    with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as handle:
        for path in sorted(experiment_path.rglob("*")):
            if not path.is_file():
                continue
            relative = path.relative_to(experiment_path)
            if not INCLUDE_GRAPH_CACHE and "processed" in relative.parts:
                skipped += 1
                continue
            handle.write(path, Path(EXPERIMENT_NAME) / relative)

    print(f"{archive}  ({archive.stat().st_size / 1e6:.1f} MB)")
    print(f"{skipped} cached graph files left out" if skipped else "Everything included")

    files.download(str(archive))   # starts a browser download

---

## 9. Deployment: predicting new polymers

This is what the models are for. `predict_external` takes a table of polymers the models have
never seen and does everything needed in one call: it prepares the structures and computes the
graphs and descriptors exactly as training did, loads **every** model in the experiment (GNN
and TML, all splits), applies the saved feature and target scalers, and combines the per-split
models into **ensembles** — one per GNN architecture, one across all GNNs, and one per TML model
× representation.

For regression an ensemble reports the mean prediction and the standard deviation across its
members, and that spread is a useful, cheap uncertainty estimate. The call also assesses the
**applicability domain** of each new polymer (section 10), and computes metrics if the target
column happens to be present — as it is here, because we held these polymers out of a labelled
dataset. For genuinely new polymers it is not, and metrics come back as `None`.

Everything is written to `unseen_predictions/<dataset>/` inside the experiment.

In [ ]:
from polynet.config.enums import ApplicabilityDomainMetric
from polynet.config.paths import unseen_predictions_experiment_parent_path
from polynet.config.schemas import ApplicabilityDomainConfig
from polynet.pipeline import predict_external

dataset_name = "new_polymers.csv"
prediction_dir = unseen_predictions_experiment_parent_path(dataset_name, experiment_path)

# Applicability domain settings. They are used by the call below, so they have to be
# decided now; section 10 explains what they mean.
ad_cfg = ApplicabilityDomainConfig(
    enabled=True,
    # A list, so both can run at once and be compared:
    #   RuzickaMorgan         - distance on Morgan fingerprints (structure)
    #   EuclideanModelInputs  - distance in the feature space each model actually sees
    metric=[ApplicabilityDomainMetric.RuzickaMorgan],
    k_neighbours=5,   # nearest training polymers used
    z=0.5,            # cutoff parameter Z in <d> + Z*sigma
    n_bins=5,         # score bins for the expected-error calibration
    # The fingerprint RuzickaMorgan measures distance with, if you want another one
    # (the default is Morgan, 2048 bins, radius 3):
    # fingerprint={"fingerprint": "rdkitfp", "fp_size": 1024},
)

new_predictions, new_metrics = predict_external(
    data=new_polymers,
    data_cfg=data_cfg,
    repr_cfg=repr_cfg,
    experiment_path=experiment_path,
    out_dir=prediction_dir,
    dataset_name=dataset_name,
    ad_cfg=ad_cfg,
)

print(f"\nPredicted {len(new_predictions)} polymers, {new_predictions.shape[1]} columns")
print("Written to:", prediction_dir)

### 9.1 The predictions

Per-split columns are named `{model}_{split} Predicted Tg`. Ensemble columns are
`{name} Ensemble Predicted Tg`, each with a companion `{name} Ensemble Std Tg` holding the
standard deviation across its members. Note the `GNN Ensemble`, which pools every GNN model
across architectures and splits.

Expect the metrics below to be worse than the test metrics from section 7, and that is not a
bug: the test sets were drawn from the same 500 polymers the models were fitted around,
while these 50 come from elsewhere in the dataset. Which is exactly the question the next
section answers properly.

In [ ]:
prediction_cols = [c for c in new_predictions.columns if ResultColumn.PREDICTED in c]
ensemble_cols = [c for c in prediction_cols if "Ensemble Predicted" in c]
std_cols = [c for c in new_predictions.columns if "Ensemble Std" in c]

print(f"Per-split prediction columns: {len(prediction_cols) - len(ensemble_cols)}")
print("Ensemble prediction columns:")
for c in ensemble_cols:
    print("  ", c)

if new_metrics is not None:
    print("\nMetrics on the unseen polymers (ensembles):")
    for model, sets in new_metrics.get("ensemble", {}).items():
        for set_name, metrics in sets.items():
            values = {str(k): round(v, 3) for k, v in metrics.items() if v is not None}
            print(f"  {model:28s} {values}")
else:
    print("\nNo target column in the new data, so no metrics - the normal case in deployment.")

show = [ResultColumn.INDEX, true_col] + ensemble_cols + std_cols
new_predictions[[c for c in show if c in new_predictions.columns]].head(10).round(3)

---

## 10. Applicability domain: when not to trust a prediction

A model will cheerfully predict a value for a polymer unlike anything it was trained on, and the
number will look just as confident as a good one. The applicability domain makes that
distinction visible, and it is the most useful thing you can report next to a prediction.

The idea is simple. For each new polymer, PolyNet measures the mean distance to its `k` nearest
**training** polymers, using how similar their Morgan fingerprints are. The training polymers
are themselves typically that far apart or closer, which sets a cutoff: a polymer inside the
cutoff is **in domain**, and one beyond it is an extrapolation. PolyNet then goes one step
further and asks what error the models actually made on **held-out test** polymers at a
comparable distance, and reports that as the **expected error** — not a theoretical bound, but
the error the model really made on polymers this unusual. That is what turns a bare number into
something reportable: *Tg ≈ 412 K, and this model was out by about 40 K on held-out polymers this far from its training set.*

Method: k-nearest-neighbour domain (Tropsha, Gramatica & Gombar, *QSAR Comb. Sci.* **2003**, 22,
69–77) with error calibration (Sheridan et al., *J. Chem. Inf. Comput. Sci.* **2004**, 44,
1912–1928). `predict_external` ran it in section 9 and added the `AD` columns to the
predictions; the per-split domains and the calibration bins are written to
`applicability_domain.json` beside them, so a referee can see exactly how a flag was produced.

**Try it:** add `ApplicabilityDomainMetric.EuclideanModelInputs` to `ad_cfg.metric` back in
section 9 for a second domain, measured in each traditional model's own scaled, feature-selected
inputs rather than in fingerprint space. It asks the sharper question: is this polymer unusual
*in the space the model actually sees*?

In [ ]:
from polynet.applicability import summarise_domain

# The index of the summary names the domain (model family or representation).
print("How many of the new polymers fall inside each domain:\n")
print(summarise_domain(new_predictions).to_string())

### 10.1 Does the domain actually predict error?

The test that matters, and the one to run on your own data before you rely on the flag.
Because these held-out polymers happen to be labelled, we can check the claim against reality:
are the polymers flagged as outside the domain really predicted worse than the ones inside it?

An empty expected error in the table below is itself informative — it means the polymer is
farther out than *every* test polymer, so no error was ever measured that far from the training
set.

Treat the *size* of the gap with caution: with a few dozen polymers, a handful of them out of
domain and only three splits, this comparison is noisy and another seed can move it a lot. What
you are looking for is the **direction**.

In [ ]:
MODEL = "random forest-rdkit"   # the model whose error estimation we check
SCOPE = "TML"                   # its domain: "GNN" or "TML"

target = data_cfg.target_variable_name
ens_col = f"{MODEL} Ensemble Predicted {target}"
exp_col = f"{MODEL} AD Ruzicka Expected Abs Error {target}"
domain_col = f"{SCOPE} AD Ruzicka In Domain"

missing = [c for c in (ens_col, exp_col, domain_col) if c not in new_predictions.columns]
if missing:
    raise KeyError(f"Adjust MODEL / SCOPE above - not in the predictions: {missing}")

audit = new_predictions[[true_col, ens_col, exp_col, domain_col]].copy()
audit["error"] = (audit[true_col] - audit[ens_col]).abs()

print(f"{MODEL}: error on the new polymers, inside and outside the domain\n")
print(
    audit.groupby(domain_col)
    .agg(
        polymers=("error", "size"),
        observed_error=("error", "mean"),
        expected_error=(exp_col, "mean"),
    )
    .round(3)
    .to_string()
)

groups, labels = [], []
for flag, label in ((True, "in domain"), (False, "out of domain")):
    values = audit.loc[audit[domain_col] == flag, "error"].dropna()
    if len(values):
        groups.append(values.to_numpy())
        labels.append(f"{label}\n(n={len(values)})")

fig, ax = plt.subplots(figsize=(5, 4.5))
ax.boxplot(groups, widths=0.55, showmeans=True)
ax.set_xticks(range(1, len(groups) + 1))
ax.set_xticklabels(labels)
ax.set_ylabel(f"absolute error{units_suffix}")
ax.set_title(f"{MODEL}: error by applicability domain")
fig.tight_layout()
plt.show()

---

## 11. Explainability

A prediction you cannot interrogate is hard to act on. If you are designing a polymer, you need
to know *which part of the structure* the model thinks matters.

### 11.1 GNNs: chemistry-masking attribution

Rather than attributing to individual atoms (hard to act on chemically), PolyNet fragments each
polymer into chemically meaningful pieces — **BRICS** retrosynthetic fragments, or the Murcko
scaffold — and masks each fragment out of the pre-pooling graph embedding:

```
attribution(fragment) = Y_predicted(full) − Y_predicted(masked)
```

A positive attribution means that fragment pushes the prediction up. This follows Wu et al.,
*Nat. Commun.* **14**, 2585 (2023).

Two views are produced: a **global** distribution over all fragments across the chosen
molecules, and **local** per-molecule atom-level heatmaps. Here we explain split 1 only and pick
two molecules from its test set.

In [ ]:
from polynet.config.enums import AttributionPlotType, FragmentationMethod
from polynet.config.schemas import ExplainabilityConfig
from polynet.pipeline import run_explainability

molecules_to_explain = [str(i) for i in test_idxs[0][:2]]

explain_cfg = ExplainabilityConfig(
    enabled=True,
    algorithm="chemistry_masking",               # the only algorithm so far
    models=["GCN"],                              # a list, or "all" for every architecture
    bootstraps=[1],                              # a list, or "all" for every split
    fragmentation=FragmentationMethod.BRICS,     # BRICS | MurckoScaffold
    explain_set="test",                          # "test" | "train" | "validation" | "all"
    normalisation="per_model",                   # per_model | local | global | no_normalisation
    plot_type=AttributionPlotType.Ridge,         # Ridge | Bar | Strip
    top_n=5,                                     # top and bottom 5 fragments; None shows all
    local_explain_mol_ids=molecules_to_explain,  # per-molecule heatmaps; None skips them
    # target_class=1,                            # classification only: the class to attribute
)

print("Explaining molecules:", molecules_to_explain)

run_explainability(
    trained_models=gnn_models,
    dataset=graph_dataset,
    split_indexes=split_indexes,
    data_cfg=data_cfg,
    exp_cfg=explain_cfg,
    out_dir=experiment_path,
)

The global plot shows which fragments systematically raise or lower the prediction. Read it as a
hypothesis generator, not a mechanism: it tells you what the *model* has learned, which is a
claim about the model and its training data, not a law of polymer physics. Rigid aromatic fragments raising Tg and flexible aliphatic chains lowering it is the sanity check to look for.

The stage saves its own `fragment_attributions.png`, but that default figure has three problems
as a figure to show other people: every row is labelled with the fragment's SMILES, which is
unreadable past a few atoms; nothing says how many observations are behind each distribution;
and a fragment seen once or twice produces a row with no visible curve at all, because there is
nothing to fit a density to.

PolyNet has the pieces to fix all three, and they are exactly what the GUI's publication export
uses: short `F` tags instead of SMILES, the observation count `n` on every row, and a separate
structure key that says which fragment each tag is. We also drop fragments seen fewer than
`MIN_OBSERVATIONS` times before taking the top and bottom five — those are the blank rows in the
default figure, and a density fitted to two points is not a distribution worth reading.

In [ ]:
from IPython.display import Image, display

from polynet.config.paths import explanation_parent_directory
from polynet.explainability import (
    FragmentGridOptions,
    compute_global_attribution,
    fragment_label_table,
    fragment_structure_png,
    ordered_attribution_fragments,
    plot_attribution_distribution,
)
from polynet.explainability.selection import match_dataset_ids, samples_per_model
from polynet.explainability.visualization import attribution_fragment_colors

TOP_N = 5             # fragments shown at each end
MIN_OBSERVATIONS = 3  # below this there is no distribution to speak of
NEGATIVE, POSITIVE = "#40bcde", "#e64747"   # PolyNet's own attribution colours

explanation_dir = explanation_parent_directory(experiment_path)

# The same models and molecules the stage explained, so this re-reads the attributions
# it cached in explanations/explanation.json and costs no further inference.
models_to_explain = {"GCN_1": gnn_models["GCN_1"]}
explain_mols = match_dataset_ids(graph_dataset, [str(i) for i in test_idxs[0]], "test set")

global_result = compute_global_attribution(
    models=models_to_explain,
    experiment_path=experiment_path,
    dataset=graph_dataset,
    explain_mols=explain_mols,
    problem_type=data_cfg.problem_type,
    normalisation_type=explain_cfg.normalisation,
    fragmentation_approach=explain_cfg.fragmentation,
    top_n=TOP_N,
    plot_type=AttributionPlotType.Ridge,
    mols_per_model=samples_per_model(models_to_explain, split_indexes, explain_cfg.explain_set),
)
plt.close(global_result.figure)   # the default figure - we redraw it ourselves below

attributions = global_result.attribution_dict
frequent = {f: s for f, s in attributions.items() if len(s) >= MIN_OBSERVATIONS}

print(f"{global_result.n_frags_total} unique fragments over {global_result.n_mols} molecules")
print(f"{len(frequent)} of them seen at least {MIN_OBSERVATIONS} times")
print(f"showing the top and bottom {TOP_N} of those")

Now the same data, labelled. `n` on each row is the number of attribution observations behind
that density — one per occurrence of the fragment in a molecule, per model — not the number of
polymers. The `F` tags are assigned over *every* fragment in the analysis, so a tag means the
same fragment whatever subset you plot, and gaps in the numbering after filtering are expected.

In [ ]:
label_table = fragment_label_table(attributions)
tag_of = dict(zip(label_table["Fragment"], label_table["Tag"]))

# Passing PolyNet's attribution colours to both figures keeps the curves, the tags and
# the structure key on one palette - this is the GUI's "match distribution" mode.
tag_colors = attribution_fragment_colors(
    frequent, AttributionPlotType.Ridge, top_n=TOP_N, neg_color=NEGATIVE, pos_color=POSITIVE
)

fig = plot_attribution_distribution(
    frequent,
    figsize=(8, 7),
    neg_color=NEGATIVE,                 # any matplotlib colour works
    pos_color=POSITIVE,
    top_n=TOP_N,                        # None draws every fragment that passed the filter
    fragment_labels=tag_of,             # None labels the rows with their SMARTS instead
    fragment_label_colors=tag_colors,
    observation_count_position="tag",   # "tag" | "axis" | None - where n goes
    # kde_bandwidth=0.3,                # widen or tighten the ridge curves
)
plt.show()

And the structure key: the same fragments, in the same order and the same colours, drawn by
RDKit. This is the second figure the GUI offers to export, and the reason the first one can
afford to be readable. `FragmentGridOptions` controls the layout — the panels here are the
GUI's own 40 × 30 mm, with one row per end of the distribution — and `dpi` is the only thing
worth raising for print.

In [ ]:
shown_fragments = ordered_attribution_fragments(frequent, TOP_N)
key_table = (
    label_table.set_index("Fragment")
    .loc[shown_fragments]
    .reset_index()[["Tag", "SMARTS", "Fragment"]]
)

structure_png = fragment_structure_png(
    key_table,
    # The GUI's own layout defaults, with one row per end of the distribution. Every
    # measurement is in millimetres, so the figure comes out at a known physical size:
    #   columns=4  mol_width_mm=40  mol_height_mm=30  horizontal_gap_mm=4
    #   vertical_gap_mm=4  bond_length_mm=5  bond_width_pt=0.7
    #   atom_font_pt=10  tag_font_pt=10
    FragmentGridOptions(columns=TOP_N),
    dpi=200,                 # 300-1200 for print; 200 keeps the notebook light
    tag_colors=tag_colors,   # None draws the structures in black
)

structure_path = explanation_dir / "fragment_structures.png"
structure_path.write_bytes(structure_png)
print("Structure key written to:", structure_path)
display(Image(filename=str(structure_path)))

The local heatmaps colour each atom by the attribution of the fragment it belongs to, which is
the most direct answer to "why did the model say *this* for *this* polymer?". The stage saves
one per molecule you asked for, alongside a CSV of the per-fragment numbers.

In [ ]:
for path in sorted(explanation_dir.glob("*_heatmap.png"))[:2]:
    print(path.name)
    display(Image(filename=str(path)))

### 11.2 Traditional models: SHAP

For descriptor-based models, PolyNet uses [SHAP](https://github.com/shap/shap), which attributes
a prediction to the input features with a game-theoretic guarantee that the contributions sum to
the prediction.

The global **beeswarm** plot is the one to learn to read: one dot per polymer per feature,
positioned by its SHAP value and coloured by the feature's value. It shows not just which
descriptors matter but *in which direction* — high values of a descriptor pushing the prediction
up give you red dots on the right. The local **waterfall** plot does the same for one polymer: it
starts at the mean prediction and shows each descriptor pushing it up or down to the final value.

We explain the RDKit representation only: SHAP over a 1024-bit fingerprint is slow and the
individual bits are hard to interpret, which is a good illustration of why descriptor choice and
explainability are linked.

**Try it:** `FragmentationMethod.MurckoScaffold` cuts polymers into ring systems rather than
retrosynthetic pieces, which suits some chemistries better. `AttributionPlotType.Bar` and `Strip`
are two other ways to read the same attributions, and `ShapGlobalPlotType.Violin` does the same
for SHAP. Section 0.6 lists them all.

In [ ]:
from polynet.config.enums import ShapGlobalPlotType
from polynet.config.schemas import TMLExplainabilityConfig
from polynet.pipeline import run_tml_explainability

tml_explain_cfg = TMLExplainabilityConfig(
    enabled=True,
    models="all",                              # or a list: ["RandomForest", "XGBoost"]
    representations=["rdkit"],                 # or "all": any descriptor set from section 3
    bootstraps=[1],                            # a list, or "all" for every split
    explain_set="test",                        # the only set TML models did not train on
    normalisation="per_model",                 # per_model | local | global | no_normalisation
    plot_type=ShapGlobalPlotType.Beeswarm,     # Beeswarm | Bar | Violin
    top_n=10,                                  # most important 10 features; None shows all
    local_explain_sample_ids=molecules_to_explain,  # None skips the per-polymer plots
    local_plot_type="waterfall",               # waterfall | force | bar
    # target_class=1,                          # classification only: the class to attribute
)

run_tml_explainability(
    tml_trained=tml_models,
    descriptor_dfs=descriptor_dfs,
    split_indexes=split_indexes,
    data_cfg=data_cfg,
    tml_exp_cfg=tml_explain_cfg,
    out_dir=experiment_path,
    validation_in_training=tml_cfg.include_validation_in_training,
)

tml_explanation_dir = explanation_dir / "tml"
for path in sorted(tml_explanation_dir.glob("*distribution*.png")):
    print(path.name)
    display(Image(filename=str(path)))

local_shap = sorted(p for p in tml_explanation_dir.glob("*.png") if "distribution" not in p.name)
for path in local_shap[:2]:
    print(path.name)
    display(Image(filename=str(path)))

### 11.3 Handing the explanations to the app

The figures above are PNGs: good for a paper, but you cannot re-pick the molecules, change the
plot style or re-colour them without running the explanation again. The GUI's **Analyse
Results** page can do all of that, and it does it without loading a single model — it reads a
small *analysis* file that holds the scores behind each figure together with the exact
selection that produced them.

The cell below writes one per figure into `explanations/analyses/`. After it runs the folder is
self-contained: zip it from section 8.2, unzip it into `~/PolyNetExperiments/`, start `polynet`
and the explanations are there to re-plot, with no training and no inference.

(PolyNet writes these from the stages themselves from version 1.0.6 on. Doing it here as well
costs nothing — the file is named after its selection, so it is rewritten, not duplicated — and
it means the folder is complete whichever release you installed in section 0.1.)

In [ ]:
from polynet.app.services.explanation_results import capture_analysis, list_saved_analyses
from polynet.config.column_names import get_predicted_label_column_name
from polynet.explainability.selection import (
    match_dataset_ids,
    samples_per_model,
    select_splits,
    split_index_of_model,
)

# The same model instances the two stages above explained. Read from the configs, so this
# keeps working when you change `models`, `representations` or `bootstraps`. Model names are
# "{architecture}_{split}" for GNNs and "{model}-{descriptor}_{split}" for the rest.
def explained_gnn(trained, config):
    splits = select_splits(config.bootstraps, N_BOOTSTRAPS)
    return {
        name: model
        for name, model in trained.items()
        if split_index_of_model(name) in splits
        and (config.models == "all" or name.rsplit("_", 1)[0] in config.models)
    }

def explained_tml(trained, config):
    splits = select_splits(config.bootstraps, N_BOOTSTRAPS)
    return {
        name: model
        for name, model in trained.items()
        if split_index_of_model(name) in splits
        and (config.models == "all" or name.split("-", 1)[0] in config.models)
        and (
            config.representations == "all"
            or name.split("-", 1)[1].rsplit("_", 1)[0] in config.representations
        )
    }

gnn_explained = explained_gnn(gnn_models, explain_cfg)
tml_explained = explained_tml(tml_models, tml_explain_cfg)

# True and predicted value per polymer, so the app can label the per-molecule panels. The
# first explained model names the column; the GNN panels read "True"/"Predicted" and SHAP
# reads lower case.
def labels(frame, model_log_name, true_key, predicted_key):
    # "GCN_1" -> column "GCN Predicted <target>", bootstrap "1". The iterator column holds
    # strings, so compare as strings.
    model, split = model_log_name.rsplit("_", 1)
    column = get_predicted_label_column_name(data_cfg.target_variable_name, model)
    rows = frame[frame[iterator].astype(str) == split].set_index(ResultColumn.INDEX)
    return {
        str(index): {true_key: row[true_col], predicted_key: row[column]}
        for index, row in rows.iterrows()
    }

gnn_labels = labels(gnn_predictions, next(iter(gnn_explained)), "True", "Predicted")
tml_labels = labels(tml_predictions, next(iter(tml_explained)), "true", "predicted")

shared_gnn = dict(
    experiment_path=experiment_path,
    models=gnn_explained,
    dataset=graph_dataset,
    problem_type=data_cfg.problem_type,
    fragmentation_approach=explain_cfg.fragmentation,
    target_class=explain_cfg.target_class,
    predictions=gnn_labels,
)
shared_tml = dict(
    experiment_path=experiment_path,
    models=tml_explained,
    descriptor_dfs=descriptor_dfs,
    problem_type=data_cfg.problem_type,
    target_class=tml_explain_cfg.target_class,
    target_col=data_cfg.target_variable_col,
    predictions=tml_labels,
)

# Graph IDs keep the type of your ID column, so match the IDs to the dataset rather than
# passing strings - the app looks the molecules up by that value when it replots.
population = match_dataset_ids(graph_dataset, [str(i) for i in test_idxs[0]], "test set")
chosen = match_dataset_ids(graph_dataset, molecules_to_explain, "molecules to explain")

# One analysis per figure: the population distribution, and the per-molecule panels.
capture_analysis("GNN", "global", {
    **shared_gnn,
    "explain_mols": population,
    "mols_per_model": samples_per_model(gnn_explained, split_indexes, explain_cfg.explain_set),
    "selection_label": f"{explain_cfg.explain_set} set",
})
capture_analysis("GNN", "local", {**shared_gnn, "explain_mols": chosen})
capture_analysis("TML", "global", {
    **shared_tml,
    "explain_sample_ids": [str(i) for i in test_idxs[0]],
    "samples_per_model": samples_per_model(tml_explained, split_indexes, tml_explain_cfg.explain_set),
    "selection_label": f"{tml_explain_cfg.explain_set} set",
})
capture_analysis("TML", "local", {**shared_tml, "explain_sample_ids": molecules_to_explain})

print(f"explained: {len(gnn_explained)} GNN and {len(tml_explained)} traditional model "
      f"instance(s); labelled {len(gnn_labels)} and {len(tml_labels)} polymers\n")
for path in list_saved_analyses(experiment_path):
    print("saved for the app:", path.relative_to(experiment_path))

---

## 12. The other two ways to run PolyNet

This notebook used the Python API, which is the right choice for scripted or customised work.
Two other entry points exist, and all three call the *same* pipeline functions, so an experiment
made with one opens in the others.

### 12.1 The graphical interface

```bash
polynet                        # opens the GUI in your browser
polynet --server.port 8502     # any `streamlit run` option is forwarded
```

A six-page Streamlit app — *Create Experiment*, *Representation*, *Train Models*, *Predict*,
*Explain Models*, *Analyse Results* — covering everything this notebook did, including the
publication figures from section 11. It reads and writes the same `~/PolyNetExperiments`
folders, so you can train here and explore in the GUI, or the other way round. The explanations
from section 11 travel with the folder too: *Analyse Results* lists each analysis this notebook
saved and replots it from the saved scores, with nothing recomputed. It is the best starting
point for colleagues who do not write Python.

(It needs a browser on the same machine, so it does not work well inside Colab — run it locally
after `pip install polynet`. An experiment you downloaded from Colab in section 8.2 has to be
unzipped into `~/PolyNetExperiments/` before the GUI will list it.)

### 12.2 The YAML pipeline

```bash
polynet run --config experiment.yaml                        # the whole pipeline, reproducibly
polynet run --config experiment.yaml --no-tml --epochs 50   # override on the command line
polynet run --config experiment.yaml --predict-data new.csv # predict a CSV after training
polynet check                                               # verify the installation
```

One file describes the whole experiment and runs every stage this notebook ran individually.
That is what you want for a cluster job, a paper's supporting information, or a run you need to
repeat in a year. Its sections map one-to-one onto the settings objects from section 2:

```yaml
experiment:      {name: "my_polymer_experiment", output_dir: "results", random_seed: 42}
data:            {data_path: "my_polymers.csv", target_variable_col: "Tg", ...}
representations: {molecular_descriptors: {rdkit: "all", morgan: true}, ...}
splitting:       {split_type: "train_val_test", n_bootstrap_iterations: 5, ...}
gnn_training:    {train_gnn: true, gnn_convolutional_layers: {GCN: {embedding_dim: 64}}}
training:        {epochs: 250}
tml_models:      {train_tml: true, selected_models: {random_forest: {n_estimators: 200}}}
explainability:  {enabled: true, fragmentation: "brics", explain_set: "test"}
prediction:      {enabled: true, data_path: "new.csv", applicability_domain: {enabled: true}}
```

The repository's
[`configs/experiment.yaml`](https://github.com/Biomaterials-for-Medical-Devices-AI/PolyNet/blob/main/configs/experiment.yaml)
is a fully commented template with every option.

---

## 13. Recap and next steps

What you ran:

1. **Loaded** a dataset — the benchmark or your own CSV — validated the structures, and looked
   at the target distribution.
2. **Configured** the experiment with typed objects that fail fast on bad input.
3. **Represented** each polymer twice — as a graph and as descriptor vectors.
4. **Split** the data repeatedly so every metric has a spread.
5. **Trained** a GNN and traditional models on identical splits, and compared them fairly.
6. **Saved** a self-contained experiment folder that the GUI and CLI can both read.
7. **Predicted** properties of unseen polymers with per-split models and ensembles.
8. **Assessed** the applicability domain, and checked that it tracks real error.
9. **Explained** the models with fragment attribution and SHAP.

### Things worth trying next

- Run it on **your own data**: set `USE_MY_OWN_DATA = True` in section 1.1 and name your
  columns. Nothing else needs editing.
- Raise `N_SAMPLES`, `N_BOOTSTRAPS` and `EPOCHS` in section 0.4 and re-run. The whole notebook
  is written to scale.
- Swap `SplitSampler.Random` for `SplitSampler.KennardStone` in section 4 and watch the test
  metrics fall — that drop is the honest measure of generalisation to new chemistry.
- Add `Network.GAT` or `Network.CGCNN` to section 5 and compare architectures.
- Add `MolecularDescriptor.PolyBERT` or `MolecularDescriptor.PolyMetriX` descriptors in
  section 3.
- Turn on automatic hyperparameter optimisation by leaving an architecture's block empty in
  section 5.
- Try a classification task: set `problem_type`, `num_classes` and `class_names`, and
  PolyNet switches metrics (accuracy, AUROC, MCC) and plots (ROC, confusion matrix) accordingly.
- Move to **copolymers** with the sibling notebook,
  [`PolyNet_workshop_copolymers.ipynb`](https://github.com/Biomaterials-for-Medical-Devices-AI/PolyNet/blob/main/notebooks/PolyNet_workshop_copolymers.ipynb):
  same workflow, plus monomer ratios.

### Where to read more

- **Repository:** <https://github.com/Biomaterials-for-Medical-Devices-AI/PolyNet>
- **Documentation:** [docs/](https://github.com/Biomaterials-for-Medical-Devices-AI/PolyNet/tree/main/docs) — `configuration.md`
  for every config option, `descriptors.md` for the representations, `explainability.md` for the
  attribution methods, `running-and-outputs.md` for the experiment folder, `api-reference.md`
  for the full API
- **Config template:** [`configs/experiment.yaml`](https://github.com/Biomaterials-for-Medical-Devices-AI/PolyNet/blob/main/configs/experiment.yaml)

### Citing PolyNet

If PolyNet contributed to work you publish, please cite the paper:

> Aguilar-Bejarano, E.; Alexander, M. R.; Winkler, D. A.; Figueredo, G. *PolyNet: A Python
> Package for Polymer Informatics.* ChemRxiv **2026**.
> doi:[10.26434/chemrxiv.15010453/v1](https://doi.org/10.26434/chemrxiv.15010453/v1)

```bibtex
@article{polynet2026,
  author  = {Aguilar-Bejarano, Eduardo and Alexander, Morgan R. and Winkler, David A.
             and Figueredo, Grazziela},
  title   = {PolyNet: A Python Package for Polymer Informatics},
  journal = {ChemRxiv},
  year    = {2026},
  doi     = {10.26434/chemrxiv.15010453/v1}
}
```

To cite the software itself as well — for the exact version you ran — use the archived release
DOI [10.5281/zenodo.23302984](https://doi.org/10.5281/zenodo.23302984), or the metadata in
[`CITATION.cff`](https://github.com/Biomaterials-for-Medical-Devices-AI/PolyNet/blob/main/CITATION.cff).

### Methods cited in this notebook

- Chemistry-masking attribution: Wu, Z. et al. *Nat. Commun.* **14**, 2585 (2023).
- Applicability domain: Tropsha, A., Gramatica, P. & Gombar, V. K. *QSAR Comb. Sci.* **22**,
  69–77 (2003).
- Expected-error calibration: Sheridan, R. P. et al. *J. Chem. Inf. Comput. Sci.* **44**,
  1912–1928 (2004).
- ¹⁹F MRI copolymer dataset: Reis, M. et al. *J. Am. Chem. Soc.* **143**, 17677–17689 (2021).
- Curated Tg dataset: via [PolyMetriX](https://github.com/lamalab-org/polymetrix).